In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.dihedrals import Dihedral
import numpy as np
import os

# Load topology and trajectory
u = mda.Universe("provide .gro", "provide .xtc")

# Define backbone dihedral atoms
backbone = {
    "alpha": ["O3'", "P", "O5'", "C5'"],
    "beta":  ["P", "O5'", "C5'", "C4'"],
    "gamma": ["O5'", "C5'", "C4'", "C3'"],
    "delta": ["C5'", "C4'", "C3'", "O3'"],
    "epsilon": ["C4'", "C3'", "O3'", "P"],
    "zeta": ["C3'", "O3'", "P", "O5'"]
}

# Ensure output folder exists
os.makedirs("dihedrals_dat", exist_ok=True)

# Compute and save backbone dihedrals
for name, atoms in backbone.items():
    groups = []
    for res in u.residues:
        sel = res.atoms.select_atoms("name " + " ".join(atoms))
        if len(sel) == 4:
            groups.append(sel)
    dih = Dihedral(groups)
    dih.run()
    np.savetxt(f"dihedrals_dat/{name}.dat", dih.angles)
    print(f"Saved {name}.dat: {dih.angles.shape}")

# Compute and save glycosidic χ dihedral
chi_groups = []
for res in u.residues:
    base = res.resname.strip()
    if base in ["A", "G"]:  # purines
        atoms = ["O4'", "C1'", "N9", "C4"]
    else:  # pyrimidines
        atoms = ["O4'", "C1'", "N1", "C2"]
    sel = res.atoms.select_atoms("name " + " ".join(atoms))
    if len(sel) == 4:
        chi_groups.append(sel)
-
dih_chi = Dihedral(chi_groups)
dih_chi.run()
np.savetxt("dihedrals_dat/chi.dat", dih_chi.angles)
print(f"Saved chi.dat: {dih_chi.angles.shape}")